# ResNet-18 country classifier on OSV-5M: Kaggle training run

One notebook version trains one condition. Settings: **Accelerator: GPU T4 x1 (or P100)**, internet off.
Inputs to attach (made by `kaggle/make_bundles.sh`):
- `geocnn-code`: code + ResNet-18 ImageNet weights
- `osv5m-c32-128`: packed 128x170 images, manifest, masks

Use **Save Version -> Save & Run All** so the run continues in the background (12 h limit).
`--time-limit-min` stops cleanly before that; to continue, attach this notebook's output as an
input of the next version and set `RESUME_FROM`.

In [ ]:
import os, shutil

CODE = "/kaggle/input/geocnn-code"      # adjust if your dataset slugs differ
DATA = "/kaggle/input/osv5m-c32-128"
WEIGHTS = f"{CODE}/resnet18-f37072fd.pth"

RUN = "ft_imagenet"   # ft_imagenet | ft_imagenet_fp | scratch | scratch_fp
FLAGS = {
    "ft_imagenet":    "--init imagenet",
    "ft_imagenet_fp": "--init imagenet --fingerprint",
    "scratch":        "--init scratch",
    "scratch_fp":     "--init scratch --fingerprint",
}[RUN]
RESUME_FROM = None    # e.g. "/kaggle/input/<this-notebook-output>/runs/ft_imagenet"
OUT = f"/kaggle/working/runs/{RUN}"
WORKERS = os.cpu_count()
os.makedirs(OUT, exist_ok=True)
if RESUME_FROM:
    for f in ("last.pt", "best.pt", "metrics.csv"):
        if os.path.exists(f"{RESUME_FROM}/{f}"):
            shutil.copy(f"{RESUME_FROM}/{f}", OUT)
print(RUN, FLAGS, "workers", WORKERS, "resume" if RESUME_FROM else "fresh")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())"
!ls -la {DATA}

Quick check (~1 min): 60 steps, prints training images/s. Skip once the setup is known to work.

In [ ]:
!python {CODE}/scripts/train.py --data-dir {DATA} --out-dir /tmp/check {FLAGS} --weights {WEIGHTS} \
    --epochs 1 --max-steps 60 --log-every 20 --limit-eval 1024 --workers {WORKERS}

Full run. Defaults (see `scripts/train.py`): ImageNet init 30 epochs, SGD lr 0.02 per 256, head lr x10; scratch 60 epochs, lr 0.2.

In [ ]:
!python {CODE}/scripts/train.py --data-dir {DATA} --out-dir {OUT} {FLAGS} --weights {WEIGHTS} \
    --workers {WORKERS} --time-limit-min 660 --resume

In [ ]:
!python {CODE}/scripts/evaluate.py --ckpt {OUT}/best.pt --features --workers {WORKERS}

In [ ]:
import json
import pandas as pd
m = pd.read_csv(f"{OUT}/metrics.csv")
display(m.tail())
ax = m.plot(x="epoch", y=["train_top1", "val_balanced_acc", "val_shared_balanced_acc"], grid=True, figsize=(7, 4))
ax.set_ylabel("accuracy")
r = json.load(open(f"{OUT}/eval/metrics.json"))
for s, v in r["splits"].items():
    print(f"{s:12s} top1 {v['top1']:.3f}  balanced {v['balanced_acc']:.3f} {v['balanced_acc_ci95']}")
print("gap test_shared - test (balanced):", r.get("gap_test_shared_minus_test"))